In [ ]:
# Code From John Tobin
# Modified by Ryan Sponzilli

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import pandas as pd
import os

def createCumulatPlotCos(datax, datay, datalabel='', title='', xlabel='', filename='test'):
    fig = plt.figure(figsize=(11,8))
    ax = fig.add_subplot(111)
    
    # get labels
    if datalabel == '':
        datalabel = [''] * len(datax)
    
    # Loop through each dataset and plot it as a step function
    for i in range(len(datax)):
        ax.step(datax[i], datay[i], label=datalabel[i])
    
    # PLOT
    ax.set_title(title, fontsize=24)
    ax.set_ylabel('Frequency', fontsize=18)
    ax.set_xlabel(xlabel, fontsize=18)
    ax.set_xlim(0, 1.0)
    ax.legend(fontsize=18)
    plt.savefig(filename + '.pdf', dpi=200)

def createCumulatPlotDeg(datax, datay, datalabel='', title='', xlabel='', filename='test'):
    fig = plt.figure(figsize=(11,8))
    ax = fig.add_subplot(111)
    
    # get labels
    if datalabel == '':
        datalabel = [''] * len(datax)
    
    # Loop through each dataset and plot it as a step function
    for i in range(len(datax)):
        ax.step(datax[i], datay[i], label=datalabel[i])
        
    # PLOT
    ax.set_title(title, fontsize=24)
    ax.set_ylabel('Frequency', fontsize=18)
    ax.set_xlabel(xlabel, fontsize=18)
    ax.set_xlim(0, 90.0)
    ax.legend(fontsize=18)
    plt.savefig(filename + '.pdf', dpi=200)

def makeCumulate(arrayData):
    # sort ascending
    sorted_pas = np.sort(arrayData, axis=None)
    # get length
    number = len(sorted_pas)
    # Compute cumulative fraction (normalized by total number of elements)
    frac_cum = (np.arange(0, number, dtype=np.float32)) / float(number)
    
    return sorted_pas, frac_cum

def perform_test(data, output_path):
    # verify output path exists
    if not os.path.exists(output_path):
        os.mkdir(output_path)

    # Create histogram plot
    fig, ax1 = plt.subplots(figsize=(11, 8))
    cosi = np.cos(data['delta_PA'][:] * np.pi / 180.0)
    bins = np.linspace(0, 1.0, 11)
    ax1.hist(cosi, bins, fill=True, facecolor='gray', alpha=0.4, histtype='bar', ec='black', linewidth=2.0)
    ax1.set_ylabel('N', fontsize=18)
    ax1.set_xlabel('cos($Delta$[Outflow-Binary])', fontsize=18)
    ax1.set_title('Outflow PA vs. Binary PA', fontsize=24)
    plt.savefig(os.path.join(output_path, 'hist.pdf'))

    # Generate cumulative distributions for observed data
    paCumulat_cos, frac_paCumulat_cos = makeCumulate(cosi)
    paCumulat_deg, frac_paCumulat_deg = makeCumulate(data['delta_PA'][:])

    # Generate random position angles and project them
    num_samples = 100000
    pa = np.random.uniform(0, 90, num_samples) # binary PA
    ofa = np.random.uniform(0, 90, num_samples) # random outflow PA offset
    inc = np.random.random_sample(num_samples) # inclination
    R = np.random.uniform(30.0, 300.0, num_samples) # binary separation distance
    projx = R * np.cos(np.radians(pa)) # project binary PA to 2D plane
    projy = R * np.sin(np.radians(pa)) * inc # project binary PA to 2D plane
    projpa = 90.0 - np.degrees(np.arctan2(projy, projx)) # expected orthogonal outflow PA
    projpa_rand = np.abs(ofa - np.degrees(np.arctan2(projy, projx))) # random outflow PA

    # Create different percentage-randomized datasets
    projpa_25p_rand = np.concatenate((projpa[:75000], projpa_rand[75000:]))
    projpa_50p_rand = np.concatenate((projpa[:50000], projpa_rand[50000:]))
    projpa_75p_rand = np.concatenate((projpa[:25000], projpa_rand[25000:]))

    # Compute cumulative distributions for models
    paCumulat_deg_model, frac_paCumulat_deg_model = makeCumulate(projpa)
    paCumulat_cos_model, frac_paCumulat_cos_model = makeCumulate(np.cos(np.radians(projpa)))
    paCumulat_deg_model_25p_rand, frac_paCumulat_deg_model_25p_rand = makeCumulate(projpa_25p_rand)
    paCumulat_cos_model_25p_rand, frac_paCumulat_cos_model_25p_rand = makeCumulate(np.cos(np.radians(projpa_25p_rand)))
    paCumulat_deg_model_50p_rand, frac_paCumulat_deg_model_50p_rand = makeCumulate(projpa_50p_rand)
    paCumulat_cos_model_50p_rand, frac_paCumulat_cos_model_50p_rand = makeCumulate(np.cos(np.radians(projpa_50p_rand)))
    paCumulat_deg_model_75p_rand, frac_paCumulat_deg_model_75p_rand = makeCumulate(projpa_75p_rand)
    paCumulat_cos_model_75p_rand, frac_paCumulat_cos_model_75p_rand = makeCumulate(np.cos(np.radians(projpa_75p_rand)))
    paCumulat_deg_model_rand, frac_paCumulat_deg_model_rand = makeCumulate(projpa_rand)
    paCumulat_cos_model_rand, frac_paCumulat_cos_model_rand = makeCumulate(np.cos(np.radians(projpa_rand)))

    # Perform statistical tests
    f = open(os.path.join(output_path, 'test_results.txt'), 'w')
    ad_tests = [
        ("AD Test (Degrees)", paCumulat_deg, paCumulat_deg_model),
        ("AD Test 25% random (Degrees)", paCumulat_deg, paCumulat_deg_model_25p_rand),
        ("AD Test 50% random (Degrees)", paCumulat_deg, paCumulat_deg_model_50p_rand),
        ("AD Test 75% random (Degrees)", paCumulat_deg, paCumulat_deg_model_75p_rand),
        ("AD Test (Cos)", paCumulat_cos, paCumulat_cos_model)
    ]
    for label, observed, model in ad_tests:
        f.write(f'{label}: {stats.anderson_ksamp([observed, model])}\n')

    ks_tests = [
        ("KS Test (Degrees)", paCumulat_deg, paCumulat_deg_model),
        ("KS Test 25% random (Degrees)", paCumulat_deg, paCumulat_deg_model_25p_rand),
        ("KS Test 50% random (Degrees)", paCumulat_deg, paCumulat_deg_model_50p_rand),
        ("KS Test 75% random (Degrees)", paCumulat_deg, paCumulat_deg_model_75p_rand),
        ("KS Test (Cos)", paCumulat_cos, paCumulat_cos_model)
    ]
    for label, observed, model in ks_tests:
        f.write(f'{label}: {stats.ks_2samp(observed, model)}\n')
    f.close()

    # Generate cumulative plots
    createCumulatPlotDeg([
        paCumulat_deg, paCumulat_deg_model, paCumulat_deg_model_rand,
        paCumulat_deg_model_25p_rand, paCumulat_deg_model_50p_rand, paCumulat_deg_model_75p_rand
    ], [
        frac_paCumulat_deg, frac_paCumulat_deg_model, frac_paCumulat_deg_model_rand,
        frac_paCumulat_deg_model_25p_rand, frac_paCumulat_deg_model_50p_rand, frac_paCumulat_deg_model_75p_rand
    ], datalabel=[
        'Observations', 'Model - Orthogonal Outflow', 'Model - Random Orientation',
        'Model - 75% Orthogonal Outflow, 25% Random',
        'Model - 50% Orthogonal Outflow, 50% Random',
        'Model - 25% Orthogonal Outflow, 75% Random'
    ], title='Outflow PA vs. Binary PA', xlabel='$\Delta$[Outflow-Binary] (degrees)', filename=os.path.join(output_path, 'OutflowPA_cumulat_deg'))

    createCumulatPlotCos([
        paCumulat_cos, paCumulat_cos_model, paCumulat_cos_model_rand
    ], [
        frac_paCumulat_cos, frac_paCumulat_cos_model, frac_paCumulat_cos_model_rand
    ], datalabel=[
        'Observations', 'Model - Orthogonal Outflow', 'Model - Random Orientation'
    ], title='Outflow PA vs. Binary PA', xlabel='cos($\Delta$[Outflow-Binary])', filename=os.path.join(output_path, 'OutflowPA_cumulat_cos'))

# script options
output_folder = "../results/stat_test"
filename = 'OutflowPA_hist'
output_path = os.path.join(output_folder, filename)

# verify output path exists
if not os.path.exists(output_folder):
    os.mkdir(output_folder)

# load data
data = pd.read_csv('../data/output/outflow_data.csv')
data = data.loc[~data['delta_PA'].isna()]

# perform_test(data.loc[data['group'] == 'orion'], os.path.join(output_folder, 'orion'))
# perform_test(data.loc[data['group'] == 'perseus'], os.path.join(output_folder, 'perseus'))
# perform_test(data, os.path.join(output_folder, 'combined'))

<>:142: SyntaxWarning: invalid escape sequence '\D'
<>:150: SyntaxWarning: invalid escape sequence '\D'
<>:142: SyntaxWarning: invalid escape sequence '\D'
<>:150: SyntaxWarning: invalid escape sequence '\D'
/var/folders/41/_gkgvhb94wd4156zplzr4cg00000gn/T/ipykernel_18133/533198446.py:142: SyntaxWarning: invalid escape sequence '\D'
  ], title='Outflow PA vs. Binary PA', xlabel='$\Delta$[Outflow-Binary] (degrees)', filename=os.path.join(output_path, 'OutflowPA_cumulat_deg'))
/var/folders/41/_gkgvhb94wd4156zplzr4cg00000gn/T/ipykernel_18133/533198446.py:150: SyntaxWarning: invalid escape sequence '\D'
  ], title='Outflow PA vs. Binary PA', xlabel='cos($\Delta$[Outflow-Binary])', filename=os.path.join(output_path, 'OutflowPA_cumulat_cos'))


In [4]:
data.loc[data['group'] == 'perseus'].loc[~data['delta_PA'].isna()]

,group,field,source_a,source_a_ra,source_a_dec,source_b,source_b_ra,source_b_dec,distance,outflow_source,red_channels,blue_channels,red_outflow_PA,blue_outflow_PA,outflow_PA,binary_PA,delta_PA
31,perseus,Per-emb-2,A,53.074729,30.829897,B,53.074717,30.829917,300.0,B,146-150,158-163,305.285,307.060,306.1725,327.264774,21.092274
32,perseus,Per-emb-5,A,52.837258,30.758386,B,52.837258,30.758386,300.0,both,141-148,164-173,297.655,298.945,298.3000,90.000000,28.300000
33,perseus,Per-emb-12,A,52.293875,31.225275,B,52.293458,31.225597,300.0,B,140-147,159-164,31.110,25.165,28.1375,307.715976,80.421524
34,perseus,Per-emb-12,A,52.293875,31.225275,B,52.293458,31.225597,300.0,A,140-147,159-164,-9.120,9.815,0.3475,307.715976,52.631524
35,perseus,Per-emb-17,A,51.912917,30.217522,B,51.912958,30.217458,300.0,both,144-148,156-162,63.650,47.650,55.6500,146.888658,88.761342
36,perseus,Per-emb-18,A,52.296946,31.308608,B,52.296917,31.308603,300.0,both,NaN,160-165,NaN,346.545,346.5450,259.215702,87.329298
37,perseus,Per-emb-22,A,51.343333,30.753669,B,51.343125,30.753650,300.0,both,138-144,155-170,124.990,132.680,128.8350,264.667841,44.167159
38,perseus,Per-emb-27,A,52.231542,31.243597,B,52.231500,31.243439,300.0,A,142-148,161-165,34.600,10.880,22.7400,194.743563,7.996437
39,perseus,Per-emb-27,A,52.231542,31.243597,B,52.231500,31.243439,300.0,B,142-149,161-166,NaN,113.920,113.9200,194.743563,80.823563
40,perseus,Per-emb-33,A,51.401333,30.754122,B,51.401292,30.754192,300.0,both,NaN,154-164,NaN,123.270,123.2700,329.036243,25.766243
